Все адреса для Армении

The goal of this notebook is to illustrate how the database with armenian addresses was initially created. As I didn't find any standartized version of list addresses I used OSM addresses to get a list of all regions, cities and streets.

In [ ]:
import requests
import pandas as pd
import time # Import time for rate limiting

# Overpass API endpoint
overpass_url = "https://overpass-api.de/api/interpreter"

# Define the Overpass Query to fetch addresses with house numbers in Armenia (ISO code AM)
# Request all relevant addr: tags and coordinates
overpass_query_new = """
[out:json][timeout:180]; // Increased timeout for larger query
area["ISO3166-1"="AM"]->.a; // Targeting the entire country of Armenia
(
  node["addr:street"]["addr:housenumber"](area.a);
  way["addr:street"]["addr:housenumber"](area.a);
  rel["addr:street"]["addr:housenumber"](area.a);
);
out tags center; // Request tags and center coordinates for ways/relations
"""

headers = {
    'Accept': 'application/json',
    'User-Agent': 'Python Overpass Client (Custom Address Script)'
}

address_data = []

try:
    print("Sending request to Overpass API...")
    response = requests.post(overpass_url, data={'data': overpass_query_new}, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    if 'application/json' in response.headers.get('Content-Type', ''):
        data_new = response.json()
        print(f"Received {len(data_new.get('elements', []))} elements from Overpass API.")

        for element in data_new.get('elements', []):
            tags = element.get('tags', {})

            osm_id = element.get('id')
            osm_type = element.get('type')

            # Latitude and Longitude extraction
            latitude = element.get('lat')
            longitude = element.get('lon')
            if (latitude is None or longitude is None) and 'center' in element:
                latitude = element['center'].get('lat')
                longitude = element['center'].get('lon')

            country_code = tags.get('addr:country')
            region = tags.get('addr:region')
            city = tags.get('addr:city')

            # Prioritize addr:suburb then addr:district for district
            district = tags.get('addr:suburb') or tags.get('addr:district')

            # Prioritize Armenian street name
            street = tags.get('addr:street:hy') or tags.get('addr:street')

            house_number = tags.get('addr:housenumber')
            postcode = tags.get('addr:postcode')

            # Construct full_address
            full_address_parts = []
            if city:
                full_address_parts.append(f"ք. {city}") # Armenian prefix for city
            if street:
                full_address_parts.append(f"{street} փ.") # Armenian suffix for street
            if house_number:
                full_address_parts.append(str(house_number))

            full_address = ", ".join(filter(None, full_address_parts))

            address_data.append({
                'osm_id': osm_id,
                'osm_type': osm_type,
                'country_code': country_code,
                'region': region,
                'city': city,
                'district': district,
                'street': street,
                'house_number': house_number,
                'postcode': postcode,
                'latitude': latitude,
                'longitude': longitude,
                'full_address': full_address
            })

        df_addresses = pd.DataFrame(address_data)
        print(f"Successfully created DataFrame with {len(df_addresses)} address entries.")

    else:
        print(f"Error: Expected JSON, but received Content-Type: {response.headers.get('Content-Type')}")
        print("Response status code:", response.status_code)
        print("Response text:\n", response.text)

except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
    if e.response is not None:
        print("Response status code:", e.response.status_code)
        print("Response text:\n", e.response.text)
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Sending request to Overpass API...
Received 119969 elements from Overpass API.
Successfully created DataFrame with 119969 address entries.


In [ ]:
data_new

{'version': 0.6,
 'generator': 'Overpass API 0.7.62.11 87bfad18',
 'osm3s': {'timestamp_osm_base': '2026-05-09T12:19:45Z',
  'timestamp_areas_base': '2026-05-09T06:04:21Z',
  'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.'},
 'elements': [{'type': 'node',
   'id': 356821190,
   'lat': 40.1840056,
   'lon': 44.5267865,
   'tags': {'access': 'private',
    'addr:city': 'Երևան',
    'addr:country': 'AM',
    'addr:housenumber': '4',
    'addr:postcode': '0025',
    'addr:street': 'Չարենցի փողոց',
    'check_date': '2025-05-04',
    'door': 'hinged',
    'entrance': 'staircase',
    'level': '0',
    'ref': '4',
    'source': 'local knowledge'}},
  {'type': 'node',
   'id': 356821194,
   'lat': 40.18401,
   'lon': 44.5265552,
   'tags': {'access': 'private',
    'addr:city': 'Երևան',
    'addr:country': 'AM',
    'addr:housenumber': '4',
    'addr:postcode': '0025',
    'addr:street': 'Չարենցի փողոց',
    'check_date':

In [ ]:
# Display the first few rows of the DataFrame
display(df_addresses.head())

# Display DataFrame information
display(df_addresses.info())

,osm_id,osm_type,country_code,region,city,district,street,house_number,postcode,latitude,longitude,full_address
0,356821190,node,AM,None,Երևան,None,Չարենցի փողոց,4,0025,40.184006,44.526787,"ք. Երևան, Չարենցի փողոց փ., 4"
1,356821194,node,AM,None,Երևան,None,Չարենցի փողոց,4,0025,40.184010,44.526555,"ք. Երևան, Չարենցի փողոց փ., 4"
2,370050884,node,AM,Երևան,Երևան,Կենտրոն,Վազգեն Սարգսյանի փողոց,26/1,0010,40.176776,44.512470,"ք. Երևան, Վազգեն Սարգսյանի փողոց փ., 26/1"
3,370050885,node,AM,Երևան,Երևան,Կենտրոն,Սայաթ-Նովայի պողոտա,19,0001,40.183503,44.519404,"ք. Երևան, Սայաթ-Նովայի պողոտա փ., 19"
4,370050886,node,AM,Երևան,Երևան,Կենտրոն,Մեսրոպ Մաշտոցի պողոտա,39/12,0002,40.188465,44.515931,"ք. Երևան, Մեսրոպ Մաշտոցի պողոտա փ., 39/12"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119969 entries, 0 to 119968
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   osm_id        119969 non-null  int64  
 1   osm_type      119969 non-null  object 
 2   country_code  78525 non-null   object 
 3   region        1010 non-null    object 
 4   city          117678 non-null  object 
 5   district      1194 non-null    object 
 6   street        119969 non-null  object 
 7   house_number  119969 non-null  object 
 8   postcode      15961 non-null   object 
 9   latitude      119969 non-null  float64
 10  longitude     119969 non-null  float64
 11  full_address  119969 non-null  object 
dtypes: float64(2), int64(1), object(9)
memory usage: 11.0+ MB


None

In [ ]:
# Save the DataFrame to a CSV file
df_addresses.to_csv("osm_am_addresses.csv", index=False)
print("DataFrame saved to 'osm_am_addresses.csv'")

DataFrame saved to 'osm_am_addresses.csv'


## Parsing of information from ena.am (the same thing was done with website veolia.am but is not represented in this notebook)

In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup



In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def parse_ena_page(url):
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for HTTP errors
    soup = BeautifulSoup(response.text, 'html.parser')

    parsed_data = []

    # 1. Extract text from paragraphs
    paragraphs = soup.find_all('p')
    for p in paragraphs:
        text = p.get_text(strip=True)
        if text:
            parsed_data.append({'type': 'paragraph', 'content': text})

    # 2. Extract content from the 'Հասցե' column in tables
    tables = soup.find_all('table')
    for table in tables:
        headers = []
        # Find headers
        for th in table.find_all('th'):
            headers.append(th.get_text(strip=True))

        address_col_index = -1
        try:
            address_col_index = headers.index('Հասցե')
        except ValueError:
            # 'Հասցե' column not found in this table, skip it
            continue

        # Find data rows
        for row in table.find_all('tr'):
            cells = row.find_all('td')
            if cells and address_col_index < len(cells):
                address_content = cells[address_col_index].get_text(strip=True)
                if address_content:
                    parsed_data.append({'type': 'table_address', 'content': address_content})

    # Create a DataFrame
    df = pd.DataFrame(parsed_data)
    return df

# URL to parse
url_to_parse = "https://www.ena.am/Info.aspx?id=5&lang=1"

# Parse the page and create DataFrame
df_parsed_content = parse_ena_page(url_to_parse)

# Display the DataFrame
print(df_parsed_content.head())
print(f"Total items extracted: {len(df_parsed_content)}")


        type                                            content
0  paragraph  «Հայաստանի էլեկտրական ցանցեր» փակ բաժնետիրական...
1  paragraph                     Երևանի Կենտրոն վարչական շրջան՝
2  paragraph  11:00-17:00Թումանյան փողոց 21, 32, 34 շենքեր, ...
3  paragraph             Երևանի Քանաքեռ Զեյթուն վարչական շրջան՝
4  paragraph  11:00-17:00Քանաքեռի 10-րդ փողոցի 4-րդ նրբանցք ...
Total items extracted: 658


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def parse_veolia_page(url):
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for HTTP errors
    soup = BeautifulSoup(response.text, 'html.parser')

    parsed_data = []

    # The page seems to have text in <p> tags and also directly within <div>s.
    # Let's try to get all block-level elements that might contain text paragraphs.
    # A common approach is to look for <p> tags, or specific <div>s/spans that hold the main content.

    # Looking at the page structure, many text blocks are within <p> tags, or <div>s with class 'field-item even'
    # or 'text_content'. Let's start with <p> tags and then refine if needed.

    # Attempt to find common content containers first
    content_divs = soup.find_all('div', class_=['text_content', 'field-item even'])

    if not content_divs:
        # Fallback if specific classes are not found, try to find all paragraphs directly
        paragraphs = soup.find_all('p')
        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                parsed_data.append({'type': 'paragraph', 'content': text})
    else:
        for div in content_divs:
            paragraphs = div.find_all('p')
            if paragraphs:
                for p in paragraphs:
                    text = p.get_text(strip=True)
                    if text:
                        parsed_data.append({'type': 'paragraph', 'content': text})
            else:
                # If no <p> tags inside the content_div, treat the div's direct text as a paragraph
                text = div.get_text(separator=' ', strip=True)
                if text:
                    parsed_data.append({'type': 'paragraph', 'content': text})

    # Create a DataFrame
    df = pd.DataFrame(parsed_data)
    return df

# URL to parse
url_veolia = "https://www.veolia.am/hy/consumers/jranjatowmner"

# Parse the page and create DataFrame
df_veolia_content = parse_veolia_page(url_veolia)

# Display the DataFrame
print(df_veolia_content.head())
print(f"Total paragraphs extracted: {len(df_veolia_content)}")


        type                                            content
0  paragraph  Ջրանջատումների հաղորդագրությունների դիտման համ...
1  paragraph  «Վեոլիա Ջուր» ընկերությունը տեղեկացնում է իր հ...
2  paragraph  Ընկերությունը հայցում է սպառողների ներողամտութ...
3  paragraph                                       15.06.2021թ.
4  paragraph  «Վեոլիա Ջուր» ընկերությունը տեղեկացնում է իր հ...
Total paragraphs extracted: 738


In [ ]:
import pandas as pd

# Concatenate the two DataFrames
df_combined_content = pd.concat([df_parsed_content, df_veolia_content], ignore_index=True)

# Display the head of the combined DataFrame
print(df_combined_content.head())

# Display the total number of rows in the combined DataFrame
print(f"Total items in combined DataFrame: {len(df_combined_content)}")


        type                                            content
0  paragraph  «Հայաստանի էլեկտրական ցանցեր» փակ բաժնետիրական...
1  paragraph                     Երևանի Կենտրոն վարչական շրջան՝
2  paragraph  11:00-17:00Թումանյան փողոց 21, 32, 34 շենքեր, ...
3  paragraph             Երևանի Քանաքեռ Զեյթուն վարչական շրջան՝
4  paragraph  11:00-17:00Քանաքեռի 10-րդ փողոցի 4-րդ նրբանցք ...
Total items in combined DataFrame: 1396


##Creating PostgreSQL database for storing parsed data

In [ ]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.2 MB/s eta 0:00:00


In [ ]:
import psycopg2
import pandas as pd

# --- UPDATE THESE VALUES ---
DB_NAME = "DB_NAME"      # Or the specific DB name you created in pgAdmin
DB_USER = "DB_USER"         # Your username
DB_PASS = "PASSWORD" 
DB_HOST = "DB_HOST" # Copy from ngrok (e.g., 0.tcp.ngrok.io)
DB_PORT = "DB_PORT"          # Copy the 5-digit number from the end of the ngrok URL
# ---------------------------

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    print("✅ Connection successful!")

    # 2. Simple test query
    cur = conn.cursor()
    cur.execute("SELECT version();")
    record = cur.fetchone()
    print(f"You are connected to: {record}")

    cur.close()
    conn.close()
    print("🔌 Connection closed.")

except Exception as e:
    print(f"❌ Error: {e}")

✅ Connection successful!
You are connected to: ('PostgreSQL 17.9 (Homebrew) on x86_64-apple-darwin23.6.0, compiled by Apple clang version 16.0.0 (clang-1600.0.26.6), 64-bit',)
🔌 Connection closed.


Saving data to a table

In [ ]:
import psycopg2

# Database connection parameters are already defined in previous cells:
# DB_NAME, DB_USER, DB_PASS, DB_HOST, DB_PORT

conn = None
cur = None

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True # For CREATE TABLE and other DDL operations
    cur = conn.cursor()
    print("✅ Connection to database established.")

    # 2. Define the CREATE TABLE SQL query
    create_table_query = """
    CREATE TABLE IF NOT EXISTS texts (
        id SERIAL PRIMARY KEY,
        text TEXT NOT NULL
    );
    """

    # 3. Execute the CREATE TABLE query
    cur.execute(create_table_query)
    print("✅ Table 'texts' created or already exists.")

    # 4. Insert data from df_combined_content into the 'texts' table
    insert_count = 0
    for index, row in df_combined_content.iterrows():
        text_content = row['content']
        insert_query = "INSERT INTO texts (text) VALUES (%s)"
        cur.execute(insert_query, (text_content,))
        insert_count += 1
    conn.commit() # Commit the changes
    print(f"✅ Successfully inserted {insert_count} rows into the 'texts' table.")

except Exception as e:
    print(f"❌ Error creating table, connecting to database, or inserting data: {e}")
finally:
    # 5. Close the cursor and connection
    if cur:
        cur.close()
    if conn:
        conn.close()
        print("🔌 Database connection closed.")


✅ Connection to database established.
✅ Table 'texts' created or already exists.
✅ Successfully inserted 1396 rows into the 'texts' table.
🔌 Database connection closed.


In [ ]:
import psycopg2

# Database connection parameters are already defined in previous cells:
# DB_NAME, DB_USER, DB_PASS, DB_HOST, DB_PORT

conn = None
cur = None

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True  # For CREATE TABLE and other DDL operations
    cur = conn.cursor()
    print("✅ Connection to database established.")

    # 2. Define the CREATE TABLE SQL query for addr_texts
    create_addr_texts_table_query = """
    CREATE TABLE IF NOT EXISTS addr_texts (
        id SERIAL PRIMARY KEY,
        text TEXT NOT NULL,
        addr_num TEXT,
        region TEXT,
        cityname TEXT,
        citytype TEXT,
        district TEXT,
        street TEXT,
        bldnum TEXT,
        bldlist TEXT
    );
    """

    # 3. Execute the CREATE TABLE query
    cur.execute(create_addr_texts_table_query)
    print("✅ Table 'addr_texts' created or already exists.")

    # 4. Insert data from 'texts' table into 'addr_texts.text'
    # Only insert if addr_texts is currently empty (or has fewer rows than texts)
    cur.execute("SELECT COUNT(*) FROM texts;")
    texts_count = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM addr_texts;")
    addr_texts_count = cur.fetchone()[0]

    if addr_texts_count < texts_count:
        print("⏳ Copying data from 'texts' to 'addr_texts'...")
        insert_from_texts_query = """
        INSERT INTO addr_texts (text)
        SELECT text FROM texts
        ON CONFLICT (id) DO NOTHING; -- Assuming you might run this multiple times and id will conflict if texts table is not reset
        """
        # If 'id' is a simple serial, direct insert will create new IDs.
        # If you need to preserve 'id' from 'texts', the schema would need to change, and we'd need to copy 'id' too.
        # For now, let's just copy the text content, allowing 'addr_texts.id' to auto-increment.

        # A simpler approach to avoid duplicates if addr_texts might have some data but needs to be filled
        # is to truncate or ensure unique text insertion.
        # For now, let's just insert all if the table is empty.

        # To insert only *new* texts without relying on ID for ON CONFLICT (as IDs will be generated for addr_texts):
        # We'll just perform a simple insert of all texts if it's not fully populated.
        cur.execute("INSERT INTO addr_texts (text) SELECT text FROM texts OFFSET %s;", (addr_texts_count,))
        conn.commit()
        print(f"✅ Successfully inserted {texts_count - addr_texts_count} new rows into 'addr_texts'.")
    elif addr_texts_count == texts_count:
        print("ℹ️ 'addr_texts' already contains all data from 'texts'. No new data inserted.")
    else:
        print("ℹ️ 'addr_texts' contains more data than 'texts'. No new data inserted based on 'texts' count.")


except Exception as e:
    print(f"❌ Error creating table or inserting data: {e}")
finally:
    # 5. Close the cursor and connection
    if cur:
        cur.close()
    if conn:
        conn.close()
        print("🔌 Database connection closed.")


✅ Connection to database established.
✅ Table 'addr_texts' created or already exists.
⏳ Copying data from 'texts' to 'addr_texts'...
✅ Successfully inserted 1396 new rows into 'addr_texts'.
🔌 Database connection closed.


Try to extract some address related information from text using regular expressions in order to have less manual work further

In [ ]:
import sys
!{sys.executable} -m pip install rapidfuzz
import psycopg2
import pandas as pd
import re
from rapidfuzz import process, fuzz

# Database connection parameters (assumed to be defined in previous cells)
# DB_NAME, DB_USER, DB_PASS, DB_HOST, DB_PORT

# Dictionary for Armenian abbreviations to full forms
ARMENIAN_ABBREVIATIONS = {
    'փ.': 'փողոց', 'փող.': 'փողոց', # street
    'պ.': 'պողոտա', 'պող.': 'պողոտա', # street
    'նրբ.': 'նրբանցք', # street
    'շ.': 'շենք', 'շենք.': 'շենք', # building, housenumber
    'տ.': 'տուն', 'տուն.': 'տուն', # building, housenumber
    'խճ.': 'խճուղի', # street
    'թաղ.': 'թաղամաս', # street or district
}

# Regex for common Armenian suffix endings for lemmatization (simple approach)
ARMENIAN_SUFFIX_PATTERNS = [
    r'ի$', r'ին$', r'ից$', r'ով$', r'ում$', r'իվ$', # Common case endings
    r'ներ$', r'իք$', r'ին$', r'ը$', r'ն$', r'ի$', r'ում$', r'ից$', r'ով$' # Added more common suffixes for normalization
]

# City type mapping for canonical forms
CITY_TYPE_MAP = {
    'քաղաք': 'քաղաք',
    'ք.': 'քաղաք',
    'գյուղ': 'գյուղ',
    'գ.': 'գյուղ'
}

# Compile suffix regex for faster processing
suffix_regex = re.compile('|'.join(ARMENIAN_SUFFIX_PATTERNS))

def normalize_text_for_matching(text):
    if not isinstance(text, str): # Handle non-string inputs like None or NaN
        return ""
    text = text.lower()
    # Replace abbreviations
    for abbr, full in ARMENIAN_ABBREVIATIONS.items():
        text = text.replace(abbr, full)
    # Remove specific punctuation that might interfere with matching
    text = re.sub(r'[.,;:`"“”\/()]', '', text)
    # Remove common suffixes (simple lemmatization)
    text = suffix_regex.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    return text

def extract_address_info(text_entry, df_addresses_normalized, fuzzy_threshold=80):
    extracted_info = {
        'cityname': None,
        'citytype': None,
        'region': None,
        'district': None,
        'street': None,
        'bldnum': None,
        'bldlist': None
    }

    original_text = text_entry
    normalized_text = normalize_text_for_matching(original_text)

    # 1. Extract City Type
    for keyword, canonical_type in CITY_TYPE_MAP.items():
        if re.search(r'\b' + re.escape(keyword) + r'(\b|ի|ին|ից|ով|ում)', original_text.lower()):
            extracted_info['citytype'] = canonical_type
            break

    # 2. Fuzzy match for City, Street, Region, District
    # Create a unique list of potential match candidates from df_addresses_normalized
    candidates_city = df_addresses_normalized['city'].dropna().unique().tolist()
    candidates_street = df_addresses_normalized['street'].dropna().unique().tolist()
    candidates_region = df_addresses_normalized['region'].dropna().unique().tolist()
    candidates_district = df_addresses_normalized['district'].dropna().unique().tolist()

    # Prioritize longer matches or more specific components

    # Try to find street first, as it's often followed by house numbers and is more specific
    best_street_match = process.extractOne(normalized_text, candidates_street, scorer=fuzz.token_set_ratio, score_cutoff=fuzzy_threshold)
    if best_street_match and best_street_match[1] >= fuzzy_threshold:
        extracted_info['street'] = best_street_match[0]

        # After finding a street, look for house numbers near it
        # Regex to find numbers that could be house numbers (single or list/range)
        # Look for numbers after a word boundary, optionally followed by '/', '-', ',', or space
        house_num_pattern = r'\b(?:\d+[\-/]?\d*(?:,\s*\d+[\-/]?\d*)*)\b'
        # Try to find house numbers that appear after the identified street, or anywhere nearby if street is vague

        # Attempt to find the street name in the original text to localize number search
        if extracted_info['street']:
            # Find the position of the street in the original text
            street_original_match = re.search(re.escape(extracted_info['street']), original_text, re.IGNORECASE)
            if street_original_match:
                search_area = original_text[street_original_match.end():] # Search after street
                # Look for 'շենք', 'տուն' (building/house) or just numbers
                bld_match = re.search(r'(?:շենք|տուն|բն.|բնակարան)?\s*(' + house_num_pattern + r')', search_area, re.IGNORECASE)
                if not bld_match:
                     # Fallback: look for numbers immediately following general address patterns
                    bld_match = re.search(r'\b(\d+[\-/]?\d*(?:,\s*\d+[\-/]?\d*)*)\b', search_area)
            else:
                # If street itself not found literally, search entire text for building numbers
                bld_match = re.search(r'\b(\d+[\-/]?\d*(?:,\s*\d+[\-/]?\d*)*)\b', original_text)
        else:
            # If no street found, search entire text for building numbers
            bld_match = re.search(r'\b(\d+[\-/]?\d*(?:,\s*\d+[\-/]?\d*)*)\b', original_text)

        if bld_match:
            numbers_str = bld_match.group(1)
            if ',' in numbers_str or '-' in numbers_str: # Check for lists or ranges
                extracted_info['bldlist'] = numbers_str
            else:
                extracted_info['bldnum'] = numbers_str

    # Then City
    best_city_match = process.extractOne(normalized_text, candidates_city, scorer=fuzz.token_set_ratio, score_cutoff=fuzzy_threshold)
    if best_city_match and best_city_match[1] >= fuzzy_threshold:
        extracted_info['cityname'] = best_city_match[0]

    # Then District
    best_district_match = process.extractOne(normalized_text, candidates_district, scorer=fuzz.token_set_ratio, score_cutoff=fuzzy_threshold)
    if best_district_match and best_district_match[1] >= fuzzy_threshold:
        extracted_info['district'] = best_district_match[0]

    # Then Region
    best_region_match = process.extractOne(normalized_text, candidates_region, scorer=fuzz.token_set_ratio, score_cutoff=fuzzy_threshold)
    if best_region_match and best_region_match[1] >= fuzzy_threshold:
        extracted_info['region'] = best_region_match[0]

    return extracted_info

# --- Main execution block ---

# Pre-process df_addresses for faster fuzzy matching
# Create a DataFrame with normalized values for fuzzy matching
df_addresses_normalized = df_addresses.copy()
for col in ['city', 'street', 'region', 'district']:
    df_addresses_normalized[col] = df_addresses_normalized[col].apply(normalize_text_for_matching)

conn = None
cur = None

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = False # Use autocommit=False to commit changes at the end
    cur = conn.cursor()
    print("✅ Connection to database established.")

    # 2. Fetch texts from addr_texts that have not yet been processed
    # Select rows where any of the address fields are NULL, to avoid reprocessing already filled rows
    cur.execute("SELECT id, text FROM addr_texts WHERE cityname IS NULL OR street IS NULL OR bldnum IS NULL OR bldlist IS NULL;")
    rows_to_process = cur.fetchall()
    print(f"Found {len(rows_to_process)} rows in 'addr_texts' to process.")

    update_count = 0
    for row_id, text_content in rows_to_process:
        if text_content:
            extracted_data = extract_address_info(text_content, df_addresses_normalized)

            # Update the addr_texts table
            update_query = """
            UPDATE addr_texts
            SET
                cityname = COALESCE(cityname, %s),
                citytype = COALESCE(citytype, %s),
                region = COALESCE(region, %s),
                district = COALESCE(district, %s),
                street = COALESCE(street, %s),
                bldnum = COALESCE(bldnum, %s),
                bldlist = COALESCE(bldlist, %s)
            WHERE id = %s;
            """
            cur.execute(update_query, (
                extracted_data['cityname'],
                extracted_data['citytype'],
                extracted_data['region'],
                extracted_data['district'],
                extracted_data['street'],
                extracted_data['bldnum'],
                extracted_data['bldlist'],
                row_id
            ))
            update_count += 1

    conn.commit() # Commit all updates at once
    print(f"✅ Successfully processed and updated {update_count} rows in 'addr_texts'.")

except Exception as e:
    print(f"❌ Error processing addresses: {e}")
    if conn:
        conn.rollback() # Rollback changes on error
        print("🔄 Transaction rolled back.")
finally:
    # 5. Close the cursor and connection
    if cur:
        cur.close()
    if conn:
        conn.close()
        print("🔌 Database connection closed.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.6 MB/s eta 0:00:00
✅ Connection to database established.
Found 1396 rows in 'addr_texts' to process.
✅ Successfully processed and updated 1396 rows in 'addr_texts'.
🔌 Database connection closed.
